<img src="img/ost_logo.png" width="240" align="right"/>
<div style="text-align: left"> <b> Applied Neural Networks: Projekt </b> <br> FS 2026 <br> <a href="mailto:ryan.mikus@ost.ch"> Ryan Mikus </a> und <a href="mailto:ilija.amato@ost.ch"> Ilija Amato </a> </div>

# Vorhersage gedämpfter Schwingungen mittels Reservoir Computing

In diesem Projekt werden gedämpfte Schwingungen mithilfe eines **Echo State Networks (ESN)** vorhergesagt. Die Schwingungen werden synthetisch erzeugt und decken verschiedene Amplituden, Dämpfungen, Frequenzen und Phasen ab. Zuerst wird eine saubere Baseline innerhalb des Trainingsbereichs erstellt. Danach wird untersucht, wie robust das Modell gegenüber Messrauschen ist und wie gut es auf Schwingungen ausserhalb des bekannten Parameterbereichs generalisiert.

Es wird Python 3.11 verwendet und die virtuelle Umgebung kann mittels requirements.txt erstellt werden.

Das Projekt ist in 6 Teile unterteilt:

1. **Einleitung**
2. [Datenerzeugung](02_Datenerzeugung.ipynb)
3. [ESN-Baseline auf sauberen Daten](03_ESN_Baseline_Clean.ipynb)
4. [Robustheit gegenüber Rauschen](04_Rauschen_Robustheit.ipynb)
5. [Out-of-Distribution-Analyse](05_Out_of_Distribution.ipynb)
6. [Schlussfolgerung](06_Schluss.ipynb)

---

# 1. Einleitung

## 1.1. Aufgabe und Anwendungsbereich

Gedämpfte Schwingungen treten in zahlreichen technischen und physikalischen Systemen auf, in Form von mechanischen Schwingungssystemen und elektrischen Schwingkreisen bis hin zu Regelkreisen und biologischen Signalen. Das präzise Vorhersagen solcher Schwingungen ist relevant für die Zustandsdiagnose, die prädiktive Regelung und die frühzeitige Erkennung von Anomalien.

In diesem Projekt wird ein **Echo State Network (ESN)** eingesetzt, um gedämpfte Schwingungen der Form

$$y(t) = A\, e^{-\alpha t} \sin(\omega t + \varphi)$$

vorherzusagen. Die Schwingungen werden synthetisch erzeugt und decken verschiedene Amplituden $A$, Dämpfungskonstanten $\alpha$, Kreisfrequenzen $\omega$ und Phasenlagen $\varphi$ ab. Gewählt wird ein **One-Step-Ahead-Prädiktionsansatz**: Das Modell sagt zum Zeitpunkt $t$ den nächsten Wert $y(t+1)$ voraus.

Neben der In-Distribution-Performance auf sauberen Testdaten wird untersucht, wie robust das Modell gegenüber Messrauschen ist und wie gut es auf Schwingungen ausserhalb des Trainingsbereichs generalisiert.

## 1.2. Motivation

Echo State Networks sind aus zwei Gründen für dieses Projekt besonders attraktiv:

**Schnelles Training:** Da das Reservoir (der rekurrente Kernteil des Netzwerks) zufällig initialisiert und anschliessend fixiert wird, muss nur die lineare Ausgabeschicht $W_{\text{out}}$ trainiert werden. Dieses Training reduziert sich auf eine lineare Regression über den gesammelten Reservoir-Zuständen und ist besonders schnell abgeschlossen, selbst auf einer normalen CPU ohne GPU-Beschleunigung. Dies ermöglicht Grid Searches mit 1000 Trainingsläufen auf Konsumerhardware.

**Eignung für dynamische Systeme:** Das rekurrente Reservoir bildet implizit ein dynamisches Gedächtnis: Es kann zeitliche Abhängigkeiten im Signal erfassen, ohne dass diese Abhängigkeiten explizit modelliert werden müssen. Gedämpfte Schwingungen sind ein klassisches Beispiel für ein dynamisches System mit zeitlicher Struktur, bei dem dieses Reservoir-Gedächtnis direkt nutzbar ist [Jaeger & Haas, 2004; Lukoševičius & Jaeger, 2009].

## 1.3. Zielsetzung und Vorgehensweise

Das Projekt verfolgt drei Teilziele:

1. **Baseline-Performance**: Wie gut kann ein ESN gedämpfte Schwingungen innerhalb des Trainingsbereichs vorhersagen?
2. **Rauschrobustheit**: Wie stabil bleibt die Vorhersage unter additivem Gauss'schem Rauschen?
3. **Generalisierung**: Wie gut generalisiert das Modell auf Out-of-Distribution-Schwingungen (grössere Amplituden, stärkere Dämpfung, höhere Frequenz)?

**Vorgehensweise:**

| Schritt | Inhalt
|---|---|
| 1 | Synthetische Daten erzeugen
| 2 | ESN-Baseline trainieren, Hyperparameter-Optimierung via Grid Search
| 3 | Rauschrobustheit untersuchen
| 4 | Out-of-Distribution-Analyse
| 5 | Zusammenfassung und Ausblick

## 1.4. Modellarchitektur: Echo State Network

### 1.4.1. Reservoir Computing

Reservoir Computing ist eine Art rekurrenter neuronaler Netze, bei dem nur die Ausgabeschicht trainiert wird. Der Kern des Netzwerks, das **Reservoir**, ist ein hochdimensionales, zufällig initialisiertes und fixiertes rekurrentes Netzwerk. Es fungiert als nichtlinearer dynamischer Projektor: Es bildet das Eingangssignal in einen hochdimensionalen Zustandsraum ab, aus dem eine trainierbare lineare Schicht die Ausgabe extrahiert [Lukoševičius & Jaeger, 2009].

Reservoir-Computing-Ansätze lassen sich in zwei Hauptvarianten unterteilen: **Echo State Networks** (ESN), eingeführt von Jaeger (2001), und **Liquid State Machines** (LSM), eingeführt von Maass et al. (2002). Beide unterscheiden sich primär in der Art der Reservoir-Neuronen (kontinuierliche tanh-Neuronen vs. Spiking-Neuronen). In diesem Projekt wird die ESN-Variante verwendet.

### 1.4.2. Architektur des ESN

Ein ESN besteht aus drei Gewichtsmatrizen:

| Komponente | Symbol | Grösse | Trainierbar |
|---|---|---|---|
| Eingangsgewichte | $W_{\text{in}}$ | $N \times 1$ | Nein (fixiert) |
| Reservoir-Gewichte | $W$ | $N \times N$ | Nein (fixiert) |
| Ausgabegewichte | $W_{\text{out}}$ | $1 \times N$ | **Ja** |

wobei $N$ die Reservoir-Grösse bezeichnet.

Der Reservoir-Zustand $\mathbf{x}(t) \in \mathbb{R}^N$ wird sequenziell aktualisiert gemäss:

$$\mathbf{x}(t+1) = (1 - \alpha)\,\mathbf{x}(t) + \alpha\, \tanh\!\left(W\,\mathbf{x}(t) + W_{\text{in}}\,u(t)\right)$$

wobei $u(t)$ der skalare Eingangswert und $\alpha \in (0, 1]$ die **Leaking Rate** ist, die die zeitliche Trägheit des Reservoirs steuert.

Die Ausgabe ist eine lineare Projektion des aktuellen Reservoir-Zustands:

$$\hat{y}(t+1) = W_{\text{out}}\,\mathbf{x}(t+1)$$

<div style="text-align: center;">
  <img src="img/diagram_ESN.png" alt="Schematische Darstellung der ESN-Architektur" style="max-width: 600px;"/>
  <br/>
  <em>Schematische Darstellung der ESN-Architektur (Input → Reservoir → Output).</em>
</div>

### 1.4.3. Echo State Property

Die **Echo State Property** (ESP) ist die zentrale Stabilitätsbedingung des ESN. Sie besagt, dass der Reservoir-Zustand $\mathbf{x}(t)$ nach einer ausreichend langen Einlaufphase (Warmup) unabhängig vom Anfangszustand $\mathbf{x}(0)$ wird. Das heisst, das Reservoir "vergisst" seinen Startzustand und antwortet ausschliesslich auf die aktuelle Eingangssequenz [Jaeger, 2001].

Eine notwendige Bedingung für die ESP ist, dass der **Spektralradius** $\rho(W)$ (der grösste Betrag der Eigenwerte von $W$) kleiner als 1 gilt. In der Praxis wird $\rho$ nahe an 1 gewählt (typisch 0.9–0.99), um ein möglichst langes Gedächtnis bei stabiler Dynamik zu erhalten [Lukoševičius, 2012]. In diesem Projekt wird $\rho$ als zentraler Hyperparameter via Grid Search optimiert.

### 1.4.4. Training

Das Training des ESN verläuft in zwei Schritten:

1. **State Collection**: Das Reservoir wird sequenziell über alle Trainingssequenzen laufen gelassen. Für jeden Zeitschritt $t$ nach der Warmup-Phase werden der Reservoir-Zustand $\mathbf{x}(t)$ und das zugehörige Ziel $y(t+1)$ gespeichert. Dies ergibt eine Zustandsmatrix $X \in \mathbb{R}^{T' \times N}$.

2. **Ausgabeschicht trainieren**: $W_{\text{out}}$ wird durch Gradientenabstieg (Adam-Optimizer) auf den gesammelten Zuständen optimiert. Das ist im Wesentlichen eine **lineare Regression** im Reservoir-Zustandsraum.

Da das Reservoir fixiert ist, müssen keine Gradienten durch die rekurrente Struktur propagiert werden. Das Training ist damit drastisch schneller als bei klassischen RNNs (LSTM, GRU) und leidet nicht unter dem Vanishing-Gradient-Problem.

### 1.4.5. Warum ESN?

ESNs wurden als Modellarchitektur aus folgenden Gründen gewählt:

- **Geschwindigkeit**: Training auf CPU in Sekunden.
- **Dynamisches Gedächtnis**: Der Reservoir-Zustand $\mathbf{x}(t)$ wird bei jedem Zeitschritt aktualisiert und trägt Informationen aus allen vorherigen Schritten mit sich. Das Modell kann damit zeitliche Abhängigkeiten im Signal erfassen und das ohne dass die Eingabedaten manuell mit einem Sliding Window aufbereitet werden müssten.
- **Theoretische Fundierung**: Der Zusammenhang zwischen Spektralradius, Gedächtnis und Stabilität ist gut verstanden [Jaeger, 2001; Lukoševičius, 2012].
- **Kein Vanishing Gradient**: Da nur $W_\text{out}$ trainiert wird und das Reservoir fixiert ist, müssen keine Gradienten durch die rekurrente Struktur propagiert werden. das bei LSTM und GRU auftretende Vanishing-Gradient-Problem entfällt.

### 1.4.6. Alternativen

Für die Vorhersage von Zeitreihen kämen alternativ folgende Architekturen in Frage:

| Modell | Vorteile | Nachteile |
|---|---|---|
| **LSTM** | Starke Kapazität für lange Abhängigkeiten | Langsames Training, viele Parameter |
| **GRU** | Einfacher als LSTM | Ähnlich langsam zu trainieren |
| **Transformer** | Parallelisierbar, sehr leistungsfähig | Hoher Datenbedarf, komplex |
| **TCN** (Temporal Convolutional Network) | Schnell, parallelisierbar | Kein echtes rekurrentes Gedächtnis |

Bei einem kleinen Datensatz (25 Trainingssequenzen) und gut verstandenen dynamischen Systemen bietet das ESN die beste Balance aus Trainingsgeschwindigkeit, Interpretierbarkeit und Eignung für zeitliche Muster.

---

<div style="display: flex; justify-content: flex-end; margin-top: 30px;">
  <a href="02_Datenerzeugung.ipynb">
    <button style="background-color:#007acc;color:white;border:none;padding:8px 12px;border-radius:6px;cursor:pointer;">
      Weiter
    </button>
  </a>
</div>